# Import libraries

In [2]:
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
import matplotlib.pyplot as plt
from IPython.display import clear_output

# Check if GPU is available for faster processing
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


# Initialize YOLOv8 and DeepSORT

In [7]:
from deep_sort_realtime.deepsort_tracker import EMBEDDER_CHOICES
# embedder can be selected below
print(EMBEDDER_CHOICES)

# Load the YOLOv8 model, It will automatically download the weights if they aren't present.
detector = YOLO('yolov8x.pt')

# Initialize the DeepSORT tracker, DeepSORT extracts appearance features (ReID) to track objects even if they get occluded.
tracker = DeepSort(
    max_age=100,               # How many frames to wait before dropping a lost track
    n_init=5,                 # Number of consecutive frames needed to confirm a track
    nms_max_overlap=0.5,      # Non-max suppression threshold
    max_cosine_distance=0.3,  # Threshold for the ReID cosine distance matching
    nn_budget=100,
    embedder="clip_ViT-B/16",     # The ReID feature extractor model
    half=False,                
    bgr=True                  # OpenCV uses BGR image format
)

# In COCO dataset (which YOLO is trained on):
# Class 0: Person
# Class 2: Car
TARGET_CLASSES = [0, 2]

# The Main Tracking Loop

In [5]:
def process_video(input_video_path, output_video_path):
    cap = cv2.VideoCapture(input_video_path)
    
    # Get video properties for saving the output
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    
    # Initialize video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    frame_count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_count += 1
        
        # Run YOLOv8 detection
        results = detector(frame, verbose=False)[0]
        
        # Format detections for DeepSORT: [ [x, y, w, h], confidence, class_id ]
        detections = []
        for box in results.boxes:
            class_id = int(box.cls.item())
            
            # Filter only for People (0) and Cars (2)
            if class_id in TARGET_CLASSES:
                confidence = float(box.conf.item())
                
                # DeepSORT expects bounding boxes in (x_center, y_center, width, height) 
                # or (left, top, width, height) depending on the method used. 
                # deep_sort_realtime expects: [left, top, w, h]
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                w = x2 - x1
                h = y2 - y1
                bbox = [x1, y1, w, h]
                
                detections.append((bbox, confidence, class_id))
                
        # Update the tracker with the new detections
        # DeepSORT crops the frame, passes it to the ReID model, and matches identities
        tracks = tracker.update_tracks(detections, frame=frame)
        
        # Draw bounding boxes and Tracking IDs
        for track in tracks:
            if not track.is_confirmed():
                continue
                
            track_id = track.track_id
            ltrb = track.to_ltrb() # Left, Top, Right, Bottom
            class_id = track.get_det_class()
            
            x1, y1, x2, y2 = map(int, ltrb)
            
            # Choose color based on class
            color = (0, 255, 0) if class_id == 0 else (255, 0, 0) # Green for people, Blue for cars
            label = f"ID: {track_id} {'Person' if class_id == 0 else 'Car'}"
            
            # Draw rectangle and text
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            
        # Write the annotated frame to the output video
        out.write(frame)
        
        # print progress
        if frame_count % 30 == 0:
            print(f"Processed {frame_count} frames...")

    cap.release()
    out.release()
    print("Video processing complete! Saved to:", output_video_path)

def process_video_ultralytics(input_path, output_path):
    # tracker="bytetrack.yaml"  
    # tracker="botsort.yaml" 
    results = model.track(
        source=input_path, 
        classes=[0, 2],         # 0: person, 2: car
        tracker="bytetrack.yaml", 
        persist=True,           # Keep tracking between frames
        save=True,              # Save the output video automatically
        project="runs",         # Directory to save the output
        name="tracking_output"  # Subfolder name
    )

# Run functions and display tracked video

In [8]:
import os
from IPython.display import Video, display

input_path = 'videos/demo_25_SY1_camera2_1.mp4'

raw_output_path = 'raw_output_tracked.mp4'
final_output_path = 'output_tracked_h264.mp4'

if os.path.exists(input_path):
    print(f"Starting tracking pipeline on: {input_path}")
    
    process_video(input_path, raw_output_path)
    os.system(f"ffmpeg -y -loglevel error -i {raw_output_path} -vcodec libx264 {final_output_path}")
    display(Video(final_output_path, embed=True, width=800))
    
else:
    print(f"Error: Could not find the video file at '{input_path}'.")
    print("Please make sure the 'videos' folder exists in your current working directory and contains the file.")

Starting tracking pipeline on: videos/demo_25_SY1_camera2_1.mp4
Processed 30 frames...
Processed 60 frames...
Processed 90 frames...
Processed 120 frames...
Processed 150 frames...
Processed 180 frames...
Processed 210 frames...
Processed 240 frames...
Processed 270 frames...
Processed 300 frames...
Processed 330 frames...
Processed 360 frames...
Processed 390 frames...
Processed 420 frames...
Processed 450 frames...
Processed 480 frames...
Processed 510 frames...
Video processing complete! Saved to: raw_output_tracked.mp4
Converting video codec for browser playback (this may take a moment)...
Done! Here is your tracked video:
